# OpenAI Realtime API — gpt-realtime-2 & gpt-realtime-whisper

Usage examples for three scripts in this folder:

1. **Text chat** — `gpt_realtime_2_example.py` (text in / text out)
2. **Streaming transcription** — `gpt_realtime_whisper_example.py` (audio in / text out)
3. **Voice turn** — `gpt_realtime_2_audio_example.py` (audio in / audio out)

## Prerequisites

```bash
pip install websockets>=14
export OPENAI_API_KEY=sk-...
```

Every live-call cell below checks for `OPENAI_API_KEY` first. If it isn't set, the cell
prints what it *would* do and skips the network call — so this notebook runs top-to-bottom
with or without a real key.


In [1]:
import asyncio
import base64
import json
import math
import os
import struct
import sys
import wave
from pathlib import Path

EXAMPLES_DIR = Path.cwd()
sys.path.insert(0, str(EXAMPLES_DIR))

import gpt_realtime_2_example as text_example
import gpt_realtime_whisper_example as whisper_example
import gpt_realtime_2_audio_example as audio_example

HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY set:", HAS_KEY)


OPENAI_API_KEY set: False


## Demo audio

The audio examples need a 24kHz / 16-bit / mono WAV file. No microphone is required for this
notebook — we synthesize a short sine-wave tone so the pipeline (chunking, base64 encoding,
`input_audio_buffer.append`/`commit`) has real bytes to send. Swap in a real recording (e.g. via
`ffmpeg -i input.mp3 -ar 24000 -ac 1 -sample_fmt s16 audio.wav`) for a meaningful transcript.


In [2]:
SAMPLE_RATE = 24000
DEMO_WAV = EXAMPLES_DIR / "demo_input.wav"

def make_demo_wav(path: Path, seconds: float = 1.5, freq: float = 440.0) -> None:
    n_samples = int(SAMPLE_RATE * seconds)
    pcm = b"".join(
        struct.pack("<h", int(3000 * math.sin(2 * math.pi * freq * i / SAMPLE_RATE)))
        for i in range(n_samples)
    )
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(SAMPLE_RATE)
        wav.writeframes(pcm)

make_demo_wav(DEMO_WAV)
print(f"Wrote demo audio: {DEMO_WAV} ({DEMO_WAV.stat().st_size} bytes)")


Wrote demo audio: C:\Users\l_ace\Desktop\projects\AIstudioAccademiaMilano\scripts\examples\realtime\demo_input.wav (72044 bytes)


## 1. Text chat — `gpt-realtime-2`

Single-turn helper built from the same `session.update` / `conversation.item.create` /
`response.create` pattern as `gpt_realtime_2_example.py` (that script's own `main()` runs an
interactive `input()` loop, which isn't notebook-friendly, so this wraps one turn as a function).


In [3]:
async def ask_once(question: str) -> str:
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    reply = []
    async with text_example.websockets.connect(text_example.URL, additional_headers=headers) as ws:
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                "output_modalities": ["text"],
                "instructions": "You are a concise, helpful assistant.",
            },
        }))
        await ws.send(json.dumps({
            "type": "conversation.item.create",
            "item": {
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": question}],
            },
        }))
        await ws.send(json.dumps({"type": "response.create", "response": {"output_modalities": ["text"]}}))

        async for raw in ws:
            event = json.loads(raw)
            if event["type"] == "response.output_text.delta":
                reply.append(event["delta"])
            elif event["type"] == "response.done":
                break
            elif event["type"] == "error":
                raise RuntimeError(event["error"])
    return "".join(reply)


if HAS_KEY:
    answer = await ask_once("What's the capital of Italy, in one sentence?")
    print(answer)
else:
    print("Skipping live call - set OPENAI_API_KEY to run ask_once() for real.")


Skipping live call - set OPENAI_API_KEY to run ask_once() for real.


## 2. Streaming transcription — `gpt-realtime-whisper`

Reuses `gpt_realtime_whisper_example.main(wav_path)` directly against the demo WAV.


In [4]:
if HAS_KEY:
    await whisper_example.main(str(DEMO_WAV))
else:
    print("Skipping live call - set OPENAI_API_KEY to run whisper_example.main() for real.")
    print(f"Would stream {DEMO_WAV} to wss://api.openai.com/v1/realtime?intent=transcription")


Skipping live call - set OPENAI_API_KEY to run whisper_example.main() for real.
Would stream C:\Users\l_ace\Desktop\projects\AIstudioAccademiaMilano\scripts\examples\realtime\demo_input.wav to wss://api.openai.com/v1/realtime?intent=transcription


## 3. Voice turn — `gpt-realtime-2` (audio in / audio out)

Reuses `gpt_realtime_2_audio_example.run_voice_turn(input_wav, output_wav)`. The model's spoken
reply is written to `reply_audio.wav` next to this notebook.


In [5]:
REPLY_WAV = EXAMPLES_DIR / "reply_audio.wav"

if HAS_KEY:
    await audio_example.run_voice_turn(str(DEMO_WAV), str(REPLY_WAV))
    print(f"Reply audio: {REPLY_WAV}")
else:
    print("Skipping live call - set OPENAI_API_KEY to run run_voice_turn() for real.")
    print(f"Would send {DEMO_WAV} and write the model's spoken reply to {REPLY_WAV}")


Skipping live call - set OPENAI_API_KEY to run run_voice_turn() for real.
Would send C:\Users\l_ace\Desktop\projects\AIstudioAccademiaMilano\scripts\examples\realtime\demo_input.wav and write the model's spoken reply to C:\Users\l_ace\Desktop\projects\AIstudioAccademiaMilano\scripts\examples\realtime\reply_audio.wav


## Notes

- `gpt-realtime-2` is speech-to-speech (no separate STT→LLM→TTS pipeline) — pick
  `output_modalities: ["text"]`, `["audio"]`, or both.
- Voices: `alloy`, `ash`, `ballad`, `coral`, `echo`, `sage`, `shimmer`, `verse`, `marin`, `cedar`
  (`marin`/`cedar` recommended for quality). The `voice` can't change after the model has spoken
  once in a session.
- `gpt-realtime-whisper` is transcription-only — it doesn't reason, call tools, or speak back.
- These examples disable `turn_detection` and commit the audio buffer manually (there's an open
  OpenAI community bug report where an explicit `turn_detection` value gets rejected on
  `gpt-realtime-whisper`).
- All audio here is raw PCM: 24kHz, 16-bit, mono, little-endian, base64-encoded over the wire.
